In [7]:
# --------------------------------------------------
# 1. Folder
# --------------------------------------------------
import meshio
import numpy as np
from pathlib import Path


input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

bulk_file = input_dir / "mesh6.vtu"
bulk = meshio.read(bulk_file)

# --------------------------------------------------
# 2. Read tetra cells and physical tags
# --------------------------------------------------
tetra = bulk.cells_dict["tetra"]
phys_tetra = bulk.cell_data_dict["gmsh:physical"]["tetra"]

# --------------------------------------------------
# 3. Physical volume tags from your table
# --------------------------------------------------
physical_volumes = {
    101: "Layer5_bottom",
    102: "Layer4_caprock2",
    103: "Layer3_Reservoir",
    104: "Layer2_caprock1",
    105: "Layer1_Top",

    201: "Well1_full",
    202: "Well1_upper",
    203: "Well1_middle",
    204: "Well1_bottom",

    205: "Well2_full",
    206: "Well2_upper",
    207: "Well2_middle",
    208: "Well2_bottom",
}


In [8]:
# --------------------------------------------------
# 4. Create one 3D VTU submesh per physical group
# --------------------------------------------------
for tag, name in physical_volumes.items():

    selected = tetra[phys_tetra == tag]

    if len(selected) == 0:
        print(f"WARNING: No tetra cells found for tag {tag} - {name}")
        continue

    used_points = np.unique(selected.flatten())
    old_to_new = {old: new for new, old in enumerate(used_points)}

    new_points = bulk.points[used_points]

    new_tetra = np.array(
        [[old_to_new[node] for node in elem] for elem in selected],
        dtype=np.int64
    )

    bulk_node_ids = used_points.astype(np.uint64)

    submesh = meshio.Mesh(
        points=new_points,
        cells=[("tetra", new_tetra)],
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={
            "gmsh:physical": [np.full(len(new_tetra), tag, dtype=np.int32)]
        }
    )

    out_file = input_dir / f"{name}.vtu"
    meshio.write(out_file, submesh)

    print(f"Created: {out_file.name}")
    print("  tag:", tag)
    print("  tetra cells:", len(new_tetra))
    print("  points:", len(new_points))

Created: Layer5_bottom.vtu
  tag: 101
  tetra cells: 27144
  points: 6139
Created: Layer4_caprock2.vtu
  tag: 102
  tetra cells: 19045
  points: 5574
Created: Layer3_Reservoir.vtu
  tag: 103
  tetra cells: 61669
  points: 13456
Created: Layer2_caprock1.vtu
  tag: 104
  tetra cells: 18926
  points: 5579
Created: Layer1_Top.vtu
  tag: 105
  tetra cells: 27269
  points: 6205
Created: Well1_full.vtu
  tag: 201
  tetra cells: 194
  points: 91
Created: Well1_upper.vtu
  tag: 202
  tetra cells: 81
  points: 44
Created: Well1_middle.vtu
  tag: 203
  tetra cells: 47
  points: 28
Created: Well1_bottom.vtu
  tag: 204
  tetra cells: 66
  points: 35
Created: Well2_full.vtu
  tag: 205
  tetra cells: 194
  points: 91
Created: Well2_upper.vtu
  tag: 206
  tetra cells: 75
  points: 42
Created: Well2_middle.vtu
  tag: 207
  tetra cells: 51
  points: 29
Created: Well2_bottom.vtu
  tag: 208
  tetra cells: 68
  points: 36


In [4]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")
bulk = meshio.read(input_dir / "mesh5.vtu")

print("Cell types:")
for cell_block in bulk.cells:
    print(cell_block.type, len(cell_block.data))

print("\nCell data keys:")
print(bulk.cell_data_dict.keys())

print("\nAvailable tetra tags:")
for key in bulk.cell_data_dict.keys():
    if "tetra" in bulk.cell_data_dict[key]:
        values = bulk.cell_data_dict[key]["tetra"]
        print("KEY:", key)
        print("Unique values:", sorted(set(values)))

Cell types:
triangle 12720
tetra 154441

Cell data keys:
dict_keys(['gmsh:physical', 'gmsh:geometrical'])

Available tetra tags:
KEY: gmsh:physical
Unique values: [np.int32(101), np.int32(102), np.int32(103), np.int32(104), np.int32(105), np.int32(201), np.int32(202)]
KEY: gmsh:geometrical
Unique values: [np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14)]


## Create MaterialIDs inside mesh6.vtu

In [12]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

mesh = meshio.read(input_dir / "mesh6.vtu")

tetra = mesh.cells_dict["tetra"]
tetra_phys = mesh.cell_data_dict["gmsh:physical"]["tetra"].astype(np.int32)

clean_mesh = meshio.Mesh(
    points=mesh.points,
    cells=[("tetra", tetra)],
    cell_data={
        "MaterialIDs": [tetra_phys],
        "gmsh:physical": [tetra_phys],
    },
)

out_file = input_dir / "mesh6_materials.vtu"
meshio.write(out_file, clean_mesh)

print("Created clean bulk mesh:", out_file)
print("Tetra MaterialIDs:", sorted(set(tetra_phys)))

Created clean bulk mesh: E:\ADATA\LITHIUM\OGS\OGS_files\input\mesh6_materials.vtu
Tetra MaterialIDs: [np.int32(101), np.int32(102), np.int32(103), np.int32(104), np.int32(105), np.int32(201), np.int32(202), np.int32(203), np.int32(204), np.int32(205), np.int32(206), np.int32(207), np.int32(208)]


In [13]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

bulk = meshio.read(input_dir / "mesh6_materials.vtu")
well1_bottom = meshio.read(input_dir / "Well1_bottom.vtu")
reservoir = meshio.read(input_dir / "Layer3_Reservoir.vtu")

bulk_ids_well = set(well1_bottom.point_data["bulk_node_ids"])
bulk_ids_res = set(reservoir.point_data["bulk_node_ids"])

shared = bulk_ids_well.intersection(bulk_ids_res)

print("Well1_bottom nodes:", len(bulk_ids_well))
print("Reservoir nodes:", len(bulk_ids_res))
print("Shared nodes:", len(shared))

Well1_bottom nodes: 35
Reservoir nodes: 13456
Shared nodes: 33


In [14]:
import pyvista as pv
from pathlib import Path

output_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\output")

result = pv.read(output_dir / "result_ts_100_t_100.000000.vtu")

pressure = result["pressure"]

print("Pressure min:", pressure.min())
print("Pressure max:", pressure.max())
print("Pressure mean:", pressure.mean())

Pressure min: -1612506.9651174129
Pressure max: 5000000.0
Pressure mean: 8319.406951386713


In [15]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

# List of surface 2D physical groups based on your Excel tables
surface_volumes = {
    301: "Left",
    302: "Right",
    303: "Front",
    304: "Back",
    305: "Top",
    306: "Bottom",
    401: "Well1_top",
    402: "Well1_completion",
    403: "Well2_top",
    404: "Well2_completion"
}

# Read bulk mesh
bulk_file = input_dir / "mesh6.vtu"
bulk = meshio.read(bulk_file)

# Loop over each surface physical tag
for tag, name in surface_volumes.items():
    # Select triangles with the given tag
    try:
        triangles = bulk.cells_dict["triangle"]
        phys_triangles = bulk.cell_data_dict["gmsh:physical"]["triangle"]
        selected = triangles[phys_triangles == tag]
    except KeyError:
        print(f"No triangle cells for tag {tag} ({name})")
        continue

    if len(selected) == 0:
        print(f"WARNING: No triangles found for tag {tag} ({name})")
        continue

    # Unique points
    used_points = np.unique(selected.flatten())
    old_to_new = {old: new for new, old in enumerate(used_points)}
    new_points = bulk.points[used_points]

    new_triangles = np.array(
        [[old_to_new[node] for node in tri] for tri in selected],
        dtype=np.int64
    )

    # bulk_node_ids for OGS BC mapping
    bulk_node_ids = used_points.astype(np.uint64)

    # Create surface mesh
    submesh = meshio.Mesh(
        points=new_points,
        cells=[("triangle", new_triangles)],
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={
            "gmsh:physical": [np.full(len(new_triangles), tag, dtype=np.int32)]
        }
    )

    # Write VTU
    out_file = input_dir / f"{name}.vtu"
    meshio.write(out_file, submesh)
    print(f"Created {out_file.name} | triangles: {len(new_triangles)} | points: {len(new_points)}")

Created Left.vtu | triangles: 1826 | points: 967
Created Right.vtu | triangles: 1830 | points: 969
Created Front.vtu | triangles: 1830 | points: 969
Created Back.vtu | triangles: 1830 | points: 969
Created Top.vtu | triangles: 2652 | points: 1400
Created Bottom.vtu | triangles: 2636 | points: 1387
Created Well1_top.vtu | triangles: 7 | points: 8
Created Well1_completion.vtu | triangles: 50 | points: 32
Created Well2_top.vtu | triangles: 7 | points: 8
Created Well2_completion.vtu | triangles: 52 | points: 33
